# Titans — a toy-scale build of neural memory learned at test time

A minimal implementation of the **neural memory module** from Behrouz,
Zhong, Mirrokni, *"Titans: Learning to Memorize at Test Time"* (2024) — the
idea that a memory can be a small neural network whose *weights themselves*
are updated on the fly, one token at a time, instead of a fixed-size vector
or matrix state.

Companion write-up: `README.md` in this folder.

## 0. Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

## 1. The idea

Every other architecture in this repo (KDA, GLA, RetNet, Mamba) represents
"memory" as a fixed-shape tensor — a matrix or vector — that gets updated by
some linear-ish update rule every step. Titans asks a different question:
**what if the memory were itself a small neural network**, and "updating the
memory" meant literally *training that network a little bit*, at every
token, while the model is running?

Concretely: the memory here is a tiny 2-layer MLP. At every timestep, three
things happen:

1. **Test the memory.** Feed the current key `k_t` into the memory MLP and
   see what it predicts. Compare that prediction to the current value `v_t`
   with a loss function — this is the model's own **associative recall
   loss**: "if I've already learned to associate keys with values, how
   wrong am I about this one?"
2. **Compute the "surprise."** Take the gradient of that loss with respect
   to the memory's own weights. This gradient *is* the "surprise" — it
   points in the direction that would make the memory less wrong about this
   particular key-value pair, right now, mid-sequence.
3. **Update the memory's weights using that gradient**, combined with a
   momentum term (so a surprising token's effect lingers a little, like
   "recent surprise") and a weight-decay term (an adaptive forget gate on
   the whole memory).

```
surprise_grad = d/dW [ loss(memory(k_t), v_t) ]        # how wrong is the memory right now
momentum = eta_t * momentum - lr_t * surprise_grad       # accumulate recent surprise
W = (1 - alpha_t) * W + momentum                         # decay old weights, apply the update
```

The output at each step is then: query the *freshly updated* memory with
the query vector `q_t`.

This is a genuinely different flavor of "state" from everything else in
this repo — instead of writing values into slots of a fixed-size tensor, the
model is doing a tiny amount of **gradient descent on itself**, live, as it
reads the sequence. `eta_t`, `alpha_t`, and the step size are all themselves
predicted from the current token, so the model controls its own learning
rate and forgetting rate on a per-token basis.

> **Simplification used here:** the paper explores several variants
> (different memory depths, different update rules, and ways to fold this
> memory into a full architecture alongside attention — "MAC", "MAG", "MAL"
> variants). This notebook implements just the core neural-memory update
> rule as a standalone sequence layer, with a shallow 2-layer MLP as the
> memory and one memory per attention head. The gradient-based update is
> made fully differentiable end-to-end using `torch.autograd.grad(...,
> create_graph=True)`, so the outer training loop can still backpropagate
> through the entire test-time learning process — this is what actually
> lets the surrounding model learn to *use* the memory well.

In [ ]:
class TitansMemory(nn.Module):
    def __init__(self, d_model=64, d_mem_hidden=16, n_heads=2, d_head=32):
        super().__init__()
        self.h, self.dh, self.dmh = n_heads, d_head, d_mem_hidden
        inner = n_heads * d_head
        self.k_proj = nn.Linear(d_model, inner, bias=False)
        self.v_proj = nn.Linear(d_model, inner, bias=False)
        self.q_proj = nn.Linear(d_model, inner, bias=False)
        self.eta_proj = nn.Linear(d_model, n_heads, bias=True)     # momentum coefficient (step 3)
        self.alpha_proj = nn.Linear(d_model, n_heads, bias=True)   # forget gate / weight decay (step 3)
        self.lr_proj = nn.Linear(d_model, n_heads, bias=True)      # surprise step size (step 3)
        self.gate_proj = nn.Linear(d_model, inner, bias=True)
        self.out_proj = nn.Linear(inner, d_model, bias=False)

    def forward(self, x):
        B, T, D = x.shape
        H, Dh, Mh = self.h, self.dh, self.dmh
        k = F.normalize(self.k_proj(x).view(B, T, H, Dh), dim=-1)   # keep memory inputs on the unit sphere --
        q = F.normalize(self.q_proj(x).view(B, T, H, Dh), dim=-1)   # this is what keeps the inner updates stable
        v = self.v_proj(x).view(B, T, H, Dh)
        eta = torch.sigmoid(self.eta_proj(x)) * 0.9                 # momentum coefficient, capped below 1
        alpha = torch.sigmoid(self.alpha_proj(x)) * 0.5 + 0.01      # forget gate, always forgets a little
        lr = torch.sigmoid(self.lr_proj(x)) * 0.1                   # small inner step size

        BH = B * H
        # the memory: a tiny 2-layer MLP, W1: Dh -> Mh, W2: Mh -> Dh, one per (batch, head)
        W1 = (torch.randn(BH, Dh, Mh, device=x.device) * 0.02).requires_grad_(True)
        W2 = (torch.randn(BH, Mh, Dh, device=x.device) * 0.02).requires_grad_(True)
        mom1 = torch.zeros_like(W1)
        mom2 = torch.zeros_like(W2)

        outs = []
        for t in range(T):
            k_t = k[:, t].reshape(BH, Dh)
            v_t = v[:, t].reshape(BH, Dh)
            q_t = q[:, t].reshape(BH, Dh)

            # step 1: test the memory -- what does it currently predict for this key?
            h_pred = torch.tanh(torch.einsum('bd,bdh->bh', k_t, W1))
            pred = torch.einsum('bh,bho->bo', h_pred, W2)
            surprise = F.mse_loss(pred, v_t, reduction='sum')       # step 1: associative recall loss

            # step 2: surprise = gradient of that loss w.r.t. the memory's own weights
            g1, g2 = torch.autograd.grad(surprise, [W1, W2], create_graph=True)
            g1, g2 = torch.clamp(g1, -1.0, 1.0), torch.clamp(g2, -1.0, 1.0)   # bound the surprise so updates can't blow up

            eta_t = eta[:, t].reshape(BH, 1, 1)
            alpha_t = alpha[:, t].reshape(BH, 1, 1)
            lr_t = lr[:, t].reshape(BH, 1, 1)

            # step 3: momentum over recent surprise, then decay + update the memory's weights
            mom1 = eta_t * mom1 - lr_t * g1
            mom2 = eta_t * mom2 - lr_t * g2
            W1 = (1 - alpha_t) * W1 + mom1
            W2 = (1 - alpha_t) * W2 + mom2

            # read out: query the freshly-updated memory
            h_out = torch.tanh(torch.einsum('bd,bdh->bh', q_t, W1))
            out_t = torch.einsum('bh,bho->bo', h_out, W2)
            outs.append(out_t.reshape(B, H, Dh))

        o = torch.stack(outs, dim=1).reshape(B, T, H * Dh)
        gate = torch.sigmoid(self.gate_proj(x))
        return self.out_proj(gate * o)

## 2. Assembling a tiny language model

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        norm = x.pow(2).mean(-1, keepdim=True)
        return x * torch.rsqrt(norm + self.eps) * self.weight

class SwiGLU(nn.Module):
    def __init__(self, d, hidden_mult=2):
        super().__init__()
        h = d * hidden_mult
        self.Wg = nn.Linear(d, h, bias=False)
        self.Wu = nn.Linear(d, h, bias=False)
        self.Wd = nn.Linear(h, d, bias=False)
    def forward(self, x):
        return self.Wd(F.silu(self.Wg(x)) * self.Wu(x))

class TinyLM(nn.Module):
    def __init__(self, vocab_size, d_model=64, n_layers=1):
        super().__init__()
        # note: 1 layer here -- Titans' per-token double-backward graph gets expensive to
        # unroll over long sequences x many layers at toy scale, so we keep depth small.
        self.embed = nn.Embedding(vocab_size, d_model)
        self.blocks = nn.ModuleList([TitansMemory(d_model) for _ in range(n_layers)])
        self.mlps = nn.ModuleList([SwiGLU(d_model) for _ in range(n_layers)])
        self.norms1 = nn.ModuleList([RMSNorm(d_model) for _ in range(n_layers)])
        self.norms2 = nn.ModuleList([RMSNorm(d_model) for _ in range(n_layers)])
        self.final_norm = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, idx):
        x = self.embed(idx)
        for blk, mlp, n1, n2 in zip(self.blocks, self.mlps, self.norms1, self.norms2):
            x = x + blk(n1(x))
            x = x + mlp(n2(x))
        return self.lm_head(self.final_norm(x))

## Proving it actually works

Everything above is only worth something if gradients actually flow correctly
through the Titans memory module once it's wired into a real model. So the rest of this
notebook:

1. wraps the Titans memory module into a tiny 2-layer causal language model,
2. builds a **tiny synthetic dataset** (a repeating `"0123456789ABCDEF"`
   string — enough to check the model can learn *any* sequential structure
   at all, no real corpus needed),
3. runs **one forward + backward pass** as a sanity check (right output
   shape, no `NaN` gradients),
4. **trains for a few hundred steps**, and
5. **generates** from the trained model — if training worked, the output
   should show visible periodicity.

This is deliberately not a "real" training run. It exists purely to catch
architecture bugs, which is the whole point of a toy-scale build.

**Note:** because every training step here involves a gradient-of-a-gradient
(the inner surprise computation is itself differentiated by the outer
optimizer), this trains noticeably slower per step than the other notebooks
in this repo, and uses a shorter sequence length to keep it fast on CPU. The
memory's inputs are also kept on the unit sphere (`F.normalize`) and the
surprise gradient is clamped before use — without both of these, the inner
weight updates compound over the sequence and can genuinely diverge to
`NaN`, since this layer is doing real (if tiny) gradient descent inside its
own forward pass.

In [ ]:
# --- synthetic dataset (shorter sequence here -- double-backward is expensive) ---
pattern = "0123456789ABCDEF"
text = pattern * 200
chars = sorted(set(text))
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}
data = torch.tensor([stoi[c] for c in text], dtype=torch.long)
vocab_size = len(chars)
max_seq_len = 16

model = TinyLM(vocab_size).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"Model built. Trainable parameters: {n_params:,}")

In [ ]:
# --- sanity check: one forward + backward pass before training ---
xb0 = data[:max_seq_len].unsqueeze(0).to(device)
yb0 = data[1:max_seq_len + 1].unsqueeze(0).to(device)
out0 = model(xb0)
logits0 = out0[0] if isinstance(out0, tuple) else out0
print(f"Sanity check -- logits shape: {tuple(logits0.shape)} (expect [1, {max_seq_len}, {vocab_size}])")
loss0 = F.cross_entropy(logits0.reshape(-1, vocab_size), yb0.reshape(-1))
if isinstance(out0, tuple):
    loss0 = loss0 + out0[1]
loss0.backward()
n_nan_grads = sum(torch.isnan(p.grad).any().item() for p in model.parameters() if p.grad is not None)
print(f"Sanity check -- initial loss: {loss0.item():.4f}, NaN grads: {n_nan_grads}")
model.zero_grad()

In [ ]:
# --- training loop (fewer steps + smaller batch: double-backward is slow on CPU) ---
def get_batch(data, block_size, batch_size, device):
    ix = torch.randint(0, len(data) - block_size - 1, (batch_size,))
    x = torch.stack([data[i:i + block_size] for i in ix])
    y = torch.stack([data[i + 1:i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

opt = torch.optim.AdamW(model.parameters(), lr=5e-3)
n_steps, batch_size = 150, 8
print("Training on synthetic periodic sequence (verifies grads flow end-to-end)...")
for step in range(n_steps):
    xb, yb = get_batch(data, max_seq_len, batch_size, device)
    logits = model(xb)
    loss = F.cross_entropy(logits.reshape(-1, vocab_size), yb.reshape(-1))
    opt.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step()
    if step % 25 == 0 or step == n_steps - 1:
        print(f"  step {step:4d} | loss {loss.item():.4f}")

In [ ]:
# --- generation ---
# Note: TitansMemory computes its update with torch.autograd.grad internally, which
# needs autograd enabled even during inference -- so unlike the other notebooks in this
# repo, we locally re-enable grad around the forward call instead of wrapping everything
# in torch.no_grad().
def generate(model, start_idx, n_new):
    model.eval()
    idx = start_idx.clone()
    for _ in range(n_new):
        with torch.enable_grad():
            logits = model(idx)
        with torch.no_grad():
            probs = F.softmax(logits[:, -1, :], dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, next_id], dim=1)
    model.train()
    return idx

start = data[:8].unsqueeze(0).to(device)
gen = generate(model, start, 48)[0].tolist()
print("Generated (should show visible periodicity if training worked):")
print(''.join(itos[i] for i in gen))

## Where to go from here

- **Try a deeper memory MLP** (more hidden layers) — the paper shows deeper
  memories can store more, at the cost of a more expensive per-token update.
- **Combine with attention.** The paper's most interesting variants (MAC,
  MAG, MAL) mix this neural memory *alongside* regular attention rather than
  using it as a full replacement — attention for precise short-range recall,
  neural memory for compressed long-range history.
- **Visualize the surprise signal** — plot `surprise` over the course of a
  sequence and see which tokens the memory finds most "surprising"; on the
  periodic toy string, you'd expect the first pass through the pattern to be
  far more surprising than later repetitions.

Reference: Behrouz, Zhong, Mirrokni, *"Titans: Learning to Memorize at Test
Time,"* 2024.